In [ ]:
#混合像素点提取
import numpy as np
import matplotlib.pyplot as plt
import spectral as spy
import json
from shapely.geometry import Polygon, Point
import pandas as pd
from collections import Counter
import os
import chardet
import warnings
warnings.filterwarnings('ignore')

# ===================== 核心配置 =====================
HDR_FILE = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl.hdr"
DAT_FILE = r"H:\图像标记\0610\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl.dat"
JSON_FILE = r"H:\图像标记\矩形标记\20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl_hres.json"
SAVE_DIR = r"H:\图像标记\矩形标记"
SAVE_FILENAME = "610_0.52矩形验证像素.xlsx"
SAVE_PATH = os.path.join(SAVE_DIR, SAVE_FILENAME)

SAVI_THRESHOLD = 0.52
RED_BAND_INDEX = 81
NIR_BAND_INDEX = 119
L = 0.5

# 1. 不一次性加载整个数据，只打开文件头（节省内存）
def open_hyperspectral_header(hdr_file):
    if not os.path.exists(hdr_file):
        raise FileNotFoundError(f"❌ HDR文件不存在：{hdr_file}")
    img = spy.open_image(hdr_file)
    print(f"✅ 高光谱文件头打开成功")
    print(f"   波段数：{img.nbands}, 行数：{img.nrows}, 列数：{img.ncols}")
    return img

# 2. 加载标注文件（兼容矩形和多边形）
def detect_file_encoding(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read()
    result = chardet.detect(raw_data)
    return result['encoding']

def load_marking_data(json_file):
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"❌ JSON文件不存在：{json_file}")
    try:
        encoding = detect_file_encoding(json_file)
        with open(json_file, 'r', encoding=encoding) as f:
            markings = json.load(f)
    except:
        with open(json_file, 'r', encoding='gbk') as f:
            markings = json.load(f)
    if 'shapes' not in markings or len(markings['shapes']) == 0:
        raise ValueError("❌ JSON文件中无有效标注数据")
    print(f"✅ 标注文件加载成功，共{len(markings['shapes'])}个标注区域")
    return markings

# 3. 核心：按需读取标注区域的像素，计算SAVI并筛选（不加载全图）
def process_markings_with_savi(img, markings):
    labeled_pixels = []
    all_savi_values = []
    soil_pixel_count = 0
    plant_pixel_count = 0
    img_height, img_width = img.nrows, img.ncols

    for idx, shape in enumerate(markings['shapes']):
        label = shape.get('label', f'label_{idx+1}')
        points = shape['points']
        
        # 矩形标注：两点生成矩形范围
        if len(points) == 2:
            x1, y1 = points[0]
            x2, y2 = points[1]
            min_x = max(0, int(min(x1, x2)))
            max_x = min(img_width-1, int(max(x1, x2)))
            min_y = max(0, int(min(y1, y2)))
            max_y = min(img_height-1, int(max(y1, y2)))
            polygon = None
        # 多边形标注：生成多边形对象
        elif len(points) >= 3:
            polygon = Polygon(points)
            min_x, min_y, max_x, max_y = polygon.bounds
            min_x = max(0, int(min_x))
            max_x = min(img_width-1, int(max_x))
            min_y = max(0, int(min_y))
            max_y = min(img_height-1, int(max_y))
        else:
            print(f"⚠️ 标注{idx+1}（{label}）点数量不足，跳过")
            continue

        shape_soil_count = 0
        shape_plant_count = 0

        # 只遍历标注区域的包围盒，不遍历全图
        for y in range(min_y, max_y + 1):
            for x in range(min_x, max_x + 1):
                # 多边形标注：判断点是否在多边形内
                if polygon is not None:
                    if not polygon.contains(Point(x, y)):
                        continue
                
                # 关键：只读取当前像素的光谱（不加载全图，节省内存）
                spectrum = img.read_pixel(y, x)
                
                # 计算SAVI
                red = float(spectrum[RED_BAND_INDEX])
                nir = float(spectrum[NIR_BAND_INDEX])
                savi = ((nir - red) / (nir + red + L + 1e-8)) * (1 + L)
                all_savi_values.append(savi)

                if savi > SAVI_THRESHOLD:
                    labeled_pixels.append((label, x, y, savi, spectrum))
                    plant_pixel_count += 1
                    shape_plant_count += 1
                else:
                    soil_pixel_count += 1
                    shape_soil_count += 1

        print(f"   标注{idx+1}（{label}）：植被像素{shape_plant_count}个，土壤像素{shape_soil_count}个")

    # SAVI统计
    if all_savi_values:
        print(f"\n📊 SAVI统计：")
        print(f"   最小值：{min(all_savi_values):.4f}")
        print(f"   最大值：{max(all_savi_values):.4f}")
        print(f"   平均值：{np.mean(all_savi_values):.4f}")
    print(f"\n🎯 筛选结果（SAVI > {SAVI_THRESHOLD}）：")
    print(f"   总计剔除土壤像素：{soil_pixel_count}个")
    print(f"   总计保留植被像素：{plant_pixel_count}个")

    return labeled_pixels

# 4. 保存结果到Excel
def save_labeled_pixels_to_excel(labeled_pixels, save_path):
    if not labeled_pixels:
        print("❌ 无符合条件的植被像素，跳过保存")
        return
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    num_bands = labeled_pixels[0][4].shape[-1]
    column_names = ['Label', 'X坐标', 'Y坐标', 'SAVI值'] + [f'Band_{i+1}' for i in range(num_bands)]
    rows = []
    for label, x, y, savi, spectrum in labeled_pixels:
        rows.append([label, x, y, round(savi, 4)] + spectrum.tolist())
    df = pd.DataFrame(rows, columns=column_names)
    df.to_excel(save_path, index=False, engine='openpyxl')
    print(f"\n✅ Excel文件保存成功：{save_path}")

# 5. 统计标签像素数量
def count_label_pixels(labeled_pixels):
    if not labeled_pixels:
        print("❌ 无像素可统计")
        return
    label_counts = Counter([label for label, _, _, _, _ in labeled_pixels])
    print("\n📈 各标签植被像素数量统计：")
    for label, count in label_counts.items():
        print(f"   {label}: {count}个")
    return label_counts

# 6. 可视化光谱曲线
def visualize_all_labeled_pixels(labeled_pixels):
    if not labeled_pixels:
        print("❌ 无植被像素可可视化")
        return
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.figure(figsize=(14, 8))
    seen_labels = set()
    label_colors = {}
    color_list = ['r', 'g', 'b', 'y', 'm', 'c', 'k', 'orange', 'purple']
    color_idx = 0
    for label, _, _, _, spectrum in labeled_pixels:
        if label not in seen_labels:
            label_colors[label] = color_list[color_idx % len(color_list)]
            color_idx += 1
            seen_labels.add(label)
        plt.plot(spectrum, color=label_colors[label], alpha=0.3,
                 label=label if label not in plt.gca().get_legend_handles_labels()[1] else "")
    plt.xlabel('波段索引', fontsize=12)
    plt.ylabel('反射率', fontsize=12)
    plt.title(f'610批次 - SAVI>0.52植被像素光谱曲线', fontsize=14)
    plt.legend(loc='best', fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# 7. 生成像素分布饼图
def generate_pie_chart(label_counts):
    if not label_counts:
        print("❌ 无数据可生成饼图")
        return
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.figure(figsize=(10, 8))
    labels = list(label_counts.keys())
    sizes = list(label_counts.values())
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#ff99cc', '#c2c2f0']
    explode = tuple([0.05] * len(labels))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=True, startangle=90, explode=explode)
    plt.title(f'610批次植被像素分布（SAVI>0.52）', fontsize=16, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    pie_chart_path = os.path.join(SAVE_DIR, "610_0.52像素分布饼图.png")
    plt.savefig(pie_chart_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ 饼图保存成功：{pie_chart_path}")
    plt.show()

# 主函数
def main():
    try:
        print("===== 开始610批次SAVI=0.52像素筛选（含坐标记录） =====")
        # 步骤1：只打开文件头，不加载全图（解决内存问题）
        img = open_hyperspectral_header(HDR_FILE)
        
        # 步骤2：加载标注文件
        markings = load_marking_data(JSON_FILE)
        
        # 步骤3：按需读取标注区域像素，计算SAVI并筛选
        print("\n📌 开始计算SAVI并筛选像素...")
        labeled_pixels = process_markings_with_savi(img, markings)
        
        # 步骤4：统计标签像素数量
        label_counts = count_label_pixels(labeled_pixels)
        
        # 步骤5：保存结果到Excel
        print("\n📌 保存筛选结果到Excel...")
        save_labeled_pixels_to_excel(labeled_pixels, SAVE_PATH)
        
        # 步骤6：可视化光谱曲线
        print("\n📌 生成光谱可视化图...")
        visualize_all_labeled_pixels(labeled_pixels)
        
        # 步骤7：生成像素分布饼图
        print("\n📌 生成像素分布饼图...")
        generate_pie_chart(label_counts)
        
        print("\n🎉 610批次SAVI像素筛选（含坐标）完成！")

    except Exception as e:
        print(f"\n❌ 程序执行失败：{str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
#像素点可视化
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator
from PIL import Image
from matplotlib.patches import Polygon

# ===============================
# 【1】路径配置
# ===============================
train_path = r"/pingtai/tanx/train_test/三类精准平衡后_训练集.xlsx"
val_path   =  r"/pingtai/tanx/train_test/filtered_points3.xlsx"
JPG_PATH = r"/pingtai/tanx/train_test/610高光谱图像.jpg"
JSON_PATH  = r"/pingtai/tanx/train_test/20250610nongdayancao_F2_2024-05-24_21-58-07-rect_refl_hres_clean.json"
SAVE_DIR   = r"/pingtai/tanx/6波段天蓝0122deletSG_XGBoost_PLSDA_矩形可视化验证1"
PADDING    = 20

os.makedirs(SAVE_DIR, exist_ok=True)

# ===============================
# 【2】固定参数
# ===============================
selected_bands = "Band_35,Band_5,Band_49,Band_31,Band_73,Band_91"

# ===============================
# 【3】SG 预处理
# ===============================
def preprocess_sg(X):
    return savgol_filter(X, window_length=5, polyorder=2, axis=1)

# ===============================
# 【4】PLS-DA 模型
# ===============================
class PLSDA(BaseEstimator):
    def __init__(self, n_components=10):
        self.n_components = n_components
        self.pls = None
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        max_comp = min(X.shape[1], X.shape[0]-1)
        self.pls = PLSRegression(n_components=min(self.n_components, max_comp))
        self.pls.fit(X, pd.get_dummies(y).values)
        return self

    def predict(self, X):
        pred = self.pls.predict(X)
        return self.classes_[np.argmax(pred, axis=1)]

# ===============================
# 【5】解析波段
# ===============================
def parse_bands(band_str):
    nums = re.findall(r'\d+', band_str)
    return [int(n)-1 for n in nums]

features = parse_bands(selected_bands)

# ===============================
# 【6】训练
# ===============================
train_df = pd.read_excel(train_path)
y_train = train_df.iloc[:,0].values
X_train = train_df.iloc[:,1:].values

X_train = X_train[:, features]
X_train = preprocess_sg(X_train)

model = PLSDA()
model.fit(X_train, y_train)

# ===============================
# 【7】验证预测
# ===============================
val_df = pd.read_excel(val_path)
x_coord = val_df["X坐标"].values
y_coord = val_df["Y坐标"].values
X_val_raw = val_df.iloc[:, 4:].values

X_val = X_val_raw[:, features]
X_val = preprocess_sg(X_val)
y_pred = model.predict(X_val)

# ===============================
# 【8】图像加载
# ===============================
img = Image.open(JPG_PATH)
W, H = img.size

with open(JSON_PATH, 'r', encoding='gbk') as f:
    json_data = json.load(f)

all_pts = []
for shape in json_data['shapes']:
    all_pts.extend(shape['points'])
all_pts = np.array(all_pts)

x_min = max(0, int(np.floor(all_pts[:, 0].min())) - PADDING)
y_min = max(0, int(np.floor(all_pts[:, 1].min())) - PADDING)
x_max = min(W, int(np.ceil(all_pts[:, 0].max())) + PADDING)
y_max = min(H, int(np.ceil(all_pts[:, 1].max())) + PADDING)

cropped = np.array(img.crop((x_min, y_min, x_max, y_max)))

mask_valid = (x_coord >= x_min) & (x_coord <= x_max) & (y_coord >= y_min) & (y_coord <= y_max)
xs = x_coord[mask_valid] - x_min
ys = y_coord[mask_valid] - y_min
ps = y_pred[mask_valid]

# ===============================
# 【9】绘图
# ===============================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 这里已经彻底修复：无空格！
def get_color(label):
    if label == 0:
        return 'red'
    elif label == 1:
        return '#FFFF00'
    else:
        return '#00C8FA'  # 绝对无空格

colors = [get_color(p) for p in ps]

# --------------------
# 图1
# --------------------
fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
ax.imshow(cropped)
ax.axis('off')
for shape in json_data['shapes']:
    pts = np.array(shape['points']) - [x_min, y_min]
    ax.add_patch(Polygon(pts, facecolor='none', edgecolor='white', linewidth=2))
ax.scatter(xs, ys, c=colors, s=6, alpha=0.8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "1_带标注区域.png"), bbox_inches='tight')
plt.close()

# --------------------
# 图2
# --------------------
fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
ax.imshow(cropped)
ax.axis('off')
ax.scatter(xs, ys, c=colors, s=6, alpha=0.8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "2_无标注区域.png"), bbox_inches='tight')
plt.close()

# --------------------
# 图3
# --------------------
plt.figure(figsize=(12, 8), dpi=300)
plt.scatter(x_coord[y_pred == 0], y_coord[y_pred == 0], c="red", s=6, alpha=0.8, label='水稻(0)')
plt.scatter(x_coord[y_pred == 1], y_coord[y_pred == 1], c="#FFFF00", s=6, alpha=0.8, label='稗草(1)')
plt.scatter(x_coord[y_pred == 2], y_coord[y_pred == 2], c="#00C8FA", s=6, alpha=0.8, label='千金子(2)')
plt.legend(prop={'size': 12})
plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "3_纯坐标散点图.png"), bbox_inches='tight')
plt.close()

# ===============================
# 完成
# ===============================
print("="*60)
print("✅ 修复完成！颜色已改为 #00C8FA 天蓝色（无空格）")
print("✅ 红色=水稻(0)，橙色=稗草(1)，天蓝=千金子(2)")
print(f"✅ 图片已保存至：{SAVE_DIR}")
print("="*60)